# 🧠 Notebook 2: Short-Term Memory con Gemini + LangChain

## ¿Qué es la memoria a corto plazo?

La **memoria a corto plazo** permite a un agente recordar el contexto de una conversación dentro de una misma sesión (thread). Sin ella, cada mensaje sería completamente independiente.

```
Sin memoria:                    Con memoria (short-term):
─────────────────               ─────────────────────────────
Usuario: "Me llamo Ana"         Usuario: "Me llamo Ana"
Agente:  "Hola, ¿en qué..."    Agente:  "Hola Ana, ¿en qué..."
                                        ↑ guarda en estado
Usuario: "¿Cómo me llamo?"     Usuario: "¿Cómo me llamo?"
Agente:  "No sé tu nombre"     Agente:  "Te llamas Ana"
                                        ↑ recupera del estado
```

**Referencia:** https://docs.langchain.com/oss/python/langchain/short-term-memory

## 📦 Instalación

In [ ]:
# !pip install -q langchain langchain-google-genai langgraph

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

## 🔴 Sin memoria: el problema

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

# Agente SIN checkpointer (sin memoria)
agente_sin_memoria = create_agent(
    model=llm,
    tools=[]  # Sin herramientas, solo conversación
)

print("=== AGENTE SIN MEMORIA ===")
print("\nTurno 1: Presentación")
r1 = agente_sin_memoria.invoke({"messages": [HumanMessage("Hola, me llamo Carlos y soy del máster de IA.")]})
print("Agente:", r1["messages"][-1].content)

print("\nTurno 2: ¿Recuerda el nombre?")
# PROBLEMA: cada invoke es independiente, no recuerda el turno anterior
r2 = agente_sin_memoria.invoke({"messages": [HumanMessage("¿Cómo me llamo?")]})
print("Agente:", r2["messages"][-1].content)

f:\Code\Evolve-MDS-2025-Oct-IAGen\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== AGENTE SIN MEMORIA ===

Turno 1: Presentación
Agente: ¡Hola Carlos! Encantado de conocerte. Es genial que estés en el máster de IA. Es un campo fascinante y con un futuro increíble.

¿En qué puedo ayudarte hoy? ¿Tienes alguna pregunta sobre IA, sobre tu máster, o simplemente quieres charlar un rato sobre el tema?

Turno 2: ¿Recuerda el nombre?
Agente: No tengo forma de saber tu nombre. Soy un modelo de lenguaje grande, entrenado por Google. No tengo acceso a información personal sobre ti.


## 🟢 Con memoria: la solución con `InMemorySaver`

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# El checkpointer guarda el estado de la conversación
checkpointer = InMemorySaver()

# Agente CON memoria a corto plazo
agente_con_memoria = create_agent(
    model=llm,
    tools=[],
    checkpointer=checkpointer  # ← Esto añade la memoria
)

# El thread_id identifica la conversación - puede haber múltiples conversaciones
config_thread1 = {"configurable": {"thread_id": "conversacion-1"}}

print("=== AGENTE CON MEMORIA (Thread 1) ===")

print("\nTurno 1: Presentación")
r1 = agente_con_memoria.invoke(
    {"messages": [HumanMessage("Hola, me llamo Carlos y soy del máster de IA.")]},
    config=config_thread1
)
print("Agente:", r1["messages"][-1].content)

print("\nTurno 2: ¿Recuerda el nombre? (mismo thread)")
r2 = agente_con_memoria.invoke(
    {"messages": [HumanMessage("¿Cómo me llamo y qué estudio?")]},
    config=config_thread1  # ← Mismo thread_id = misma conversación
)
print("Agente:", r2["messages"][-1].content)

=== AGENTE CON MEMORIA (Thread 1) ===

Turno 1: Presentación
Agente: ¡Hola Carlos! Encantado de conocerte. Es genial que estés en el máster de IA. Es un campo fascinante y con un futuro increíble.

¿En qué puedo ayudarte hoy? ¿Tienes alguna pregunta sobre IA, sobre tu máster, o simplemente quieres charlar un rato sobre el tema?

Turno 2: ¿Recuerda el nombre? (mismo thread)
Agente: Según lo que me has dicho, te llamas **Carlos** y estudias el **máster de IA**.


## 🔀 Múltiples threads: conversaciones independientes

In [ ]:
# Thread 2: diferente usuario, diferente conversación
config_thread2 = {"configurable": {"thread_id": "conversacion-2"}}

print("=== THREAD 2 (conversación diferente) ===")

r_thread2 = agente_con_memoria.invoke(
    {"messages": [HumanMessage("Hola, me llamo María.")]},
    config=config_thread2
)
print("María - Turno 1:", r_thread2["messages"][-1].content)

# Carlos en su propio thread sigue siendo Carlos
r_carlos_again = agente_con_memoria.invoke(
    {"messages": [HumanMessage("¿Me recuerdas?")]},
    config=config_thread1  # ← Thread de Carlos
)
print("\nCarlos en Thread 1 - ¿Me recuerdas?:")
print("Agente:", r_carlos_again["messages"][-1].content)

=== THREAD 2 (conversación diferente) ===
María - Turno 1: ¡Hola, María! Encantado de conocerte. ¿En qué puedo ayudarte hoy?

Carlos en Thread 1 - ¿Me recuerdas?:
Agente: Sí, **te recuerdo perfectamente**. Me dijiste que te llamas **Carlos** y que estás cursando el **máster de IA**.

¿Hay algo más en lo que pueda ayudarte o algo que quieras que recuerde?


## 📊 Inspeccionar el estado del thread

In [ ]:
# Podemos ver todos los mensajes guardados en el thread
estado = agente_con_memoria.get_state(config_thread1)

print("📋 Historial del Thread 1 (Carlos):")
print(f"   Total de mensajes: {len(estado.values['messages'])}\n")

for i, msg in enumerate(estado.values["messages"], 1):
    rol = "👤 Usuario" if msg.__class__.__name__ == "HumanMessage" else "🤖 Agente"
    print(f"  [{i}] {rol}: {msg.content[:100]}")

📋 Historial del Thread 1 (Carlos):
   Total de mensajes: 6

  [1] 👤 Usuario: Hola, me llamo Carlos y soy del máster de IA.
  [2] 🤖 Agente: ¡Hola Carlos! Encantado de conocerte. Es genial que estés en el máster de IA. Es un campo fascinante
  [3] 👤 Usuario: ¿Cómo me llamo y qué estudio?
  [4] 🤖 Agente: Según lo que me has dicho, te llamas **Carlos** y estudias el **máster de IA**.
  [5] 👤 Usuario: ¿Me recuerdas?
  [6] 🤖 Agente: Sí, **te recuerdo perfectamente**. Me dijiste que te llamas **Carlos** y que estás cursando el **más


## ✂️ Gestión de conversaciones largas: Trim Messages

Cuando la conversación crece demasiado, puede exceder el contexto del LLM. La solución: truncar mensajes antiguos.

In [ ]:
from langchain.agents import AgentState
from langchain_core.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime
from typing import Any

# Middleware que mantiene solo los últimos N mensajes
def trim_to_last_n_messages(n: int = 4):
    """Crea un middleware que mantiene solo los últimos N mensajes."""

    def trim(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        messages = state["messages"]

        if len(messages) <= n:
            return None  # No necesita recorte

        print(f"  ✂️ Recortando: {len(messages)} → {n} mensajes")

        # Quedarse solo con los últimos N mensajes
        recent = messages[-n:]
        return {
            "messages": [
                RemoveMessage(id=REMOVE_ALL_MESSAGES),
                *recent
            ]
        }

    return trim

# NOTA: En la API actual de LangChain, el middleware se pasa como lista
# Este patrón funciona con la API de create_react_agent extendida
print("✅ Función de trim definida")
print("💡 En producción, se usa como middleware en create_agent()")
print("   Ver: https://docs.langchain.com/oss/python/langchain/short-term-memory#trim-messages")

✅ Función de trim definida
💡 En producción, se usa como middleware en create_agent()
   Ver: https://docs.langchain.com/oss/python/langchain/short-term-memory#trim-messages


## 📝 Resumen: ¿Cuándo usar qué estrategia de memoria?

| Estrategia | Cuándo usarla | Ventaja | Desventaja |
|------------|---------------|---------|------------|
| **Sin recorte** | Conversaciones cortas | Contexto completo | Puede exceder ventana |
| **Trim** | Conversaciones largas simples | Rápido y simple | Pierde contexto antiguo |
| **Summarize** | Conversaciones largas importantes | Retiene info clave | Más lento (llamada extra al LLM) |
| **Delete** | Limpiar mensajes específicos | Control granular | Requiere lógica manual |

## 🧪 Ejercicio propuesto

1. Crea una conversación de al menos 6 turnos sobre un tema
2. Inspecciona el estado (`get_state`) para ver todos los mensajes
3. Implementa manualmente un trim que solo conserve el primer y los últimos 2 mensajes
4. Bonus: Crea 3 threads diferentes con 3 "usuarios" distintos